In [1]:
import pandas as pd
from Levenshtein import distance
import numpy as np
import re

In [2]:
df_csv = pd.read_excel('2025_01_27_Liste der Entschädigungsämter mit Adressen, Varianten und Archiven.xlsx')
df_csv.head()

,Layout class,Filename,Bundesland,CompensationOffice,Zuständiges Archiv,Zuständiges Archiv - GND-Nummer,Unnamed: 6
0,ABC-Karten,1930_12_13_41_0,Niedersachsen,Stadt Celle,Niedersächsisches Landesarchiv,6510041-4,NaN
1,ABC-Karten,1901_03_02_64_0,Niedersachsen,Kreissonderhilfsausschuß \nDelmenhorst,Niedersächsisches Landesarchiv,6510041-4,NaN
2,ABC-Karten,1907_12_31_115_0,Niedersachsen,Goslar,Niedersächsisches Landesarchiv,6510041-4,NaN
3,ABC-Karten,1909_01_06_66_0,Niedersachsen,Goslar. Stadt,Niedersächsisches Landesarchiv,6510041-4,NaN
4,ABC-Karten,1904_01_08_54_0,Niedersachsen,Göttingen-Stadt,Niedersächsisches Landesarchiv,6510041-4,NaN


In [3]:
df_json = pd.read_json('compoff_extracted.jsonl', lines=True)
df_json.head()

,CompensationOffice1,BZKNr,filename
0,Regierungsbezirksamt für Wiedergutmachung und ...,65477,1875_00_00_101_0.jpg
1,Landesamt für die Wiedergutmachung Karlsruhe,EK 20093 A,1875_04_05_16_0.jpg
2,Niedersachsen,1 27574 c,1875_06_15_14_0.jpg
3,Bezirksamt für Wiedergutmachung Neustadt a.d.W.,200 488,1875_04_15_1_0.jpg
4,Entschädigungsamt Berlin,70 002,1875_07_12_9_0.jpg


In [4]:
print(len(df_csv))
print(len(df_json))

n = df_json['CompensationOffice1'].isin(df_csv['CompensationOffice']).sum()
print(f'Number of matches: {n}')

424
1901284
Number of matches: 1121989


# Matching based on Lvenshtein Distance

In [5]:
compensation_offices = list(df_csv['CompensationOffice'])

# remove linebreaks and sequences of whitespaces
def remove_linebreak(s):
    s = str(s).replace('\n', ' ').replace('\r', ' ')
    return re.sub(r'\s+', ' ', s).strip().lower()

# return the minimum distance and the best match up to a threshold of half the length of the cleaned name
def get_min_distance_dynamic_normalized(name):
    name_clean = remove_linebreak(name)
    threshold = len(name_clean) // 2
    
    min_dist = float('inf')
    best_match = None
    for office in compensation_offices:
        office_clean = remove_linebreak(office)
        d = distance(name_clean, office_clean)
        if d == 0:
            return 0, office
        if d < min_dist:
            min_dist = d
            best_match = office
    if min_dist <= threshold:
        return min_dist, best_match
    return np.nan, None

results = df_json['CompensationOffice1'].apply(get_min_distance_dynamic_normalized)

df_matches = pd.DataFrame({
    'CompensationOffice1': df_json['CompensationOffice1'].apply(remove_linebreak),
    'MatchedOffice': results.apply(lambda x: x[1]),
    'MatchDistance': results.apply(lambda x: x[0])
})

df_matches.head(20)

,CompensationOffice1,MatchedOffice,MatchDistance
0,regierungsbezirksamt für wiedergutmachung und ...,Regierungsbezirksamt \nfür Wiedergutmachung \n...,0.0
1,landesamt für die wiedergutmachung karlsruhe,Landesamt für die Wiedergutmachung \nKarlsruhe,0.0
2,niedersachsen,Niedersachsen,0.0
3,bezirksamt für wiedergutmachung neustadt a.d.w.,Bezirksamt \nfür Wiedergutmachung \nNeustadt a...,2.0
4,entschädigungsamt berlin,Entschädigungsamt Berlin,0.0
5,entschädigungsamt berlin,Entschädigungsamt Berlin,0.0
6,köln,Köln,0.0
7,amt für wiedergutmachung des selbstkantkreises...,Amt für Wiedergutmachung des Selfkantkreises G...,9.0
8,landesentschädigungsamt schleswig-holstein in ...,Landesentschädigungsamt\nSchleswig-Holstein in...,0.0
9,hessen kassel,Hessen Kassel,0.0


In [6]:
print(f"Max MatchDistance: {df_matches['MatchDistance'].max()}")

Max MatchDistance: 67.0


In [7]:
total = len(df_matches)
max_dist = int(df_matches['MatchDistance'].max()) + 1
for i in range(max_dist):
    count = (df_matches['MatchDistance'] == i).sum()

    pct = count / total * 100    
    print(f'MatchDistance = {i}: {count} ({pct:.2f}%)')

MatchDistance = 0: 1422538 (74.82%)
MatchDistance = 1: 64054 (3.37%)
MatchDistance = 2: 24334 (1.28%)
MatchDistance = 3: 22113 (1.16%)
MatchDistance = 4: 43303 (2.28%)
MatchDistance = 5: 10644 (0.56%)
MatchDistance = 6: 24848 (1.31%)
MatchDistance = 7: 6033 (0.32%)
MatchDistance = 8: 5534 (0.29%)
MatchDistance = 9: 21285 (1.12%)
MatchDistance = 10: 4700 (0.25%)
MatchDistance = 11: 4947 (0.26%)
MatchDistance = 12: 10438 (0.55%)
MatchDistance = 13: 3892 (0.20%)
MatchDistance = 14: 2818 (0.15%)
MatchDistance = 15: 2491 (0.13%)
MatchDistance = 16: 2054 (0.11%)
MatchDistance = 17: 2153 (0.11%)
MatchDistance = 18: 2218 (0.12%)
MatchDistance = 19: 5044 (0.27%)
MatchDistance = 20: 1473 (0.08%)
MatchDistance = 21: 1185 (0.06%)
MatchDistance = 22: 13153 (0.69%)
MatchDistance = 23: 1518 (0.08%)
MatchDistance = 24: 17725 (0.93%)
MatchDistance = 25: 582 (0.03%)
MatchDistance = 26: 1000 (0.05%)
MatchDistance = 27: 269 (0.01%)
MatchDistance = 28: 242 (0.01%)
MatchDistance = 29: 635 (0.03%)
MatchDista

In [8]:
df_nomatches = df_matches[(df_matches['MatchDistance'] > 0) | (df_matches['MatchDistance'].isna())]

df_nomatches.sort_values('MatchDistance', ascending=True).to_excel('compensation_office_nomatches.xlsx', index=False)

# Matching without Punctuation

In [9]:
compensation_offices = list(df_csv['CompensationOffice'])

# remove linebreaks, punctuation and sequences of whitespaces
def remove_linebreak_punctuation(s):
    s = str(s).replace('\n', ' ').replace('\r', ' ')
    p = ['?', '!', '.', ',', ';', ':', '-', '_', '(', ')', '[', ']', '{', '}', '"', "'", '/']
    for char in p:
        s = s.replace(char, ' ')
    return re.sub(r'\s+', ' ', s).strip().lower()

#updated version of get_min_distance_dynamic_normalized that also removes punctuation
def get_min_distance_dynamic_normalized(name):
    name_clean = remove_linebreak_punctuation(name)
    threshold = len(name_clean) // 2
    
    min_dist = float('inf')
    best_match = None
    for office in compensation_offices:
        office_clean = remove_linebreak_punctuation(office)
        d = distance(name_clean, office_clean)
        if d == 0:
            return 0, office
        if d < min_dist:
            min_dist = d
            best_match = office
    if min_dist <= threshold:
        return min_dist, best_match
    return np.nan, None

results = df_json['CompensationOffice1'].apply(get_min_distance_dynamic_normalized)

df_matches = pd.DataFrame({
    'CompensationOffice1': df_json['CompensationOffice1'].apply(remove_linebreak_punctuation),
    'MatchedOffice': results.apply(lambda x: x[1]),
    'MatchDistance': results.apply(lambda x: x[0])
})

df_matches.head(20)

,CompensationOffice1,MatchedOffice,MatchDistance
0,regierungsbezirksamt für wiedergutmachung und ...,Regierungsbezirksamt \nfür Wiedergutmachung \n...,0.0
1,landesamt für die wiedergutmachung karlsruhe,Landesamt für die Wiedergutmachung \nKarlsruhe,0.0
2,niedersachsen,Niedersachsen,0.0
3,bezirksamt für wiedergutmachung neustadt a d w,Bezirksamt \nfür Wiedergutmachung \nNeustadt a...,0.0
4,entschädigungsamt berlin,Entschädigungsamt Berlin,0.0
5,entschädigungsamt berlin,Entschädigungsamt Berlin,0.0
6,köln,Köln,0.0
7,amt für wiedergutmachung des selbstkantkreises...,Amt für Wiedergutmachung des Selfkantkreises G...,7.0
8,landesentschädigungsamt schleswig holstein in ...,Landesentschädigungsamt\nSchleswig-Holstein in...,0.0
9,hessen kassel,Hessen Kassel,0.0


In [10]:
total = len(df_matches)
max_dist = int(df_matches['MatchDistance'].max()) + 1
for i in range(max_dist):
    count = (df_matches['MatchDistance'] == i).sum()

    pct = count / total * 100    
    print(f'MatchDistance = {i}: {count} ({pct:.2f}%)')

MatchDistance = 0: 1474765 (77.57%)
MatchDistance = 1: 33750 (1.78%)
MatchDistance = 2: 10717 (0.56%)
MatchDistance = 3: 19463 (1.02%)
MatchDistance = 4: 43260 (2.28%)
MatchDistance = 5: 9386 (0.49%)
MatchDistance = 6: 23139 (1.22%)
MatchDistance = 7: 5620 (0.30%)
MatchDistance = 8: 6111 (0.32%)
MatchDistance = 9: 22174 (1.17%)
MatchDistance = 10: 3724 (0.20%)
MatchDistance = 11: 3588 (0.19%)
MatchDistance = 12: 10586 (0.56%)
MatchDistance = 13: 3822 (0.20%)
MatchDistance = 14: 2880 (0.15%)
MatchDistance = 15: 2658 (0.14%)
MatchDistance = 16: 2481 (0.13%)
MatchDistance = 17: 2541 (0.13%)
MatchDistance = 18: 2527 (0.13%)
MatchDistance = 19: 15769 (0.83%)
MatchDistance = 20: 2457 (0.13%)
MatchDistance = 21: 1501 (0.08%)
MatchDistance = 22: 1215 (0.06%)
MatchDistance = 23: 18172 (0.96%)
MatchDistance = 24: 1059 (0.06%)
MatchDistance = 25: 569 (0.03%)
MatchDistance = 26: 233 (0.01%)
MatchDistance = 27: 528 (0.03%)
MatchDistance = 28: 949 (0.05%)
MatchDistance = 29: 1741 (0.09%)
MatchDistan

In [11]:
df_no_matches_no_punctuation = df_matches[(df_matches['MatchDistance'] > 0) | (df_matches['MatchDistance'].isna())]
df_no_matches_no_punctuation = df_no_matches_no_punctuation.sort_values('MatchDistance', ascending=False)
df_no_matches_no_punctuation.head(20)

,CompensationOffice1,MatchedOffice,MatchDistance
422246,der regierungspräsident dezernat für wiedergut...,Der Regierungspräsident \n- Dezernat für Wiede...,64.0
47910,freie und hansestadt hamburg behörde für arbei...,"Behörde für Arbeit, Gesunheit und Soziales - ...",61.0
227493,regierungsbezirksamt für wiedergutmachung und ...,Regierungsbezirksamt für Wiedergutmachung und ...,60.0
1135308,der regierungspräsident dezernat für wiedergut...,Der Regierungspräsident \n- Dezernat für Wiede...,60.0
1881670,freie und hansestadt hamburg behörde für arbei...,"Behörde für Arbeit, Gesunheit und Soziales - ...",60.0
1617342,freie und hansestadt hamburg behörde für arbei...,"Behörde für Arbeit, Gesunheit und Soziales - ...",60.0
1138927,der regierungspräsident dezernat für wiedergut...,Der Regierungspräsident \n- Dezernat für Wiede...,60.0
1126699,der regierungspräsident – dezernat für wiederg...,Der Regierungspräsident \n- Dezernat für Wiede...,60.0
293761,freie und hansestadt hamburg behörde für arbei...,"Behörde für Arbeit, Gesunheit und Soziales - ...",60.0
1827229,der regierungspräsident dezernat für wiedergut...,Der Regierungspräsident \n- Dezernat für Wiede...,60.0


In [ ]:
df_no_matches_no_punctuation.to_excel('compensation_office_nomatches_no_punctuation.xlsx', index=False)

# No Matches

In [12]:
no_matched_office_df = df_nomatches[df_nomatches['MatchDistance'].isna()]

no_matched_office_df

,CompensationOffice1,MatchedOffice,MatchDistance
21,hess. staatsministerium,NaN,NaN
30,nan,NaN,NaN
32,freie und hansestadt hamburg,NaN,NaN
34,hess. staatsministerium,NaN,NaN
40,nan,NaN,NaN
...,...,...,...
1901203,reg.-präsident / präsident des verw.-bezirks h...,NaN,NaN
1901238,hng,NaN,NaN
1901254,reg.-präsident /präsident des vereinsbezirks h...,NaN,NaN
1901262,bayrisches landesamt für verfolgtenwesen,NaN,NaN


In [13]:
no_matched_office_df_set = set(no_matched_office_df['CompensationOffice1'])

len(no_matched_office_df_set)
for office in no_matched_office_df_set:
    print(office)


6.7.28
w8in
r.a. rumpel
st. h. 6664
bmf - v34
ostalbisch-land
hessisches ministerium für jugend, familie und gesundheit
sondervermögens- und bauförderungsbehörde
feststellungsbeschv. stuttgart erhalten
rheinland-pfälzische landesregierung, neustadt a.d. weinstr.
st. h. 1665/1966 h
niedergmachungsämter von berlin
ksha düsseldorf-stadt
betty ludwig rogert albert goetz irma herbert
bayerisches landesstelle
holzminden
niedersachsen-präsidium / präsident des verw.-bezirks braunschweig
sachverständigenstelle für befristete reg.
n
länderzentrale für die regelung von schadensersatzansprüchen
reg.-präsident / präsident des verw.-bezirks-hildesheim
deutscher pellmeldung
sta
tr. bayrisches landesarchiv
kshä winsen/l.
st. h. 2050
usmf
freiburg, karlsruhe
niedersachsen / reg.-präsident / präsident des verw.-bezirks hildesheim
nentwich
bu, verw, amt
namenloses bayerisches versicherungsamt
bayrisches landesamt für verfolgte des nationalsozialismus
kreissonderhilfsausschübe
e 2742
landesarchiv für ba

# Calling LLM via API using NaN values

In [14]:
from groq import Groq
from huggingface_hub import InferenceClient
from huggingface_hub.utils import HfHubHTTPError

import time
import json
import os
from dotenv import load_dotenv
from tqdm import tqdm


/home/Benha/Projects/Arbeit/FIZ/compensation-office-analysis/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
load_dotenv()

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "groq").strip().lower()
HF_MODEL = os.getenv("HF_MODEL", "Qwen/Qwen3-8B")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.1-8b-instant")

# choosing between Hugging Face and Groq based on environment variable, standard = groq
if LLM_PROVIDER == "huggingface":
    HF_API_TOKEN = os.getenv("HF_API_KEY")
    if not HF_API_TOKEN:
        raise ValueError("HF_API_KEY is required when LLM_PROVIDER='huggingface'")
    client = InferenceClient(
        model=HF_MODEL,
        token=HF_API_TOKEN
    )
    ACTIVE_MODEL = HF_MODEL
elif LLM_PROVIDER == "groq":
    GROQ_API_KEY = os.getenv("GROQ_API_KEY")
    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY is required when LLM_PROVIDER='groq'")
    client = Groq(api_key=GROQ_API_KEY)
    ACTIVE_MODEL = GROQ_MODEL
else:
    raise ValueError("LLM_PROVIDER must be either 'huggingface' or 'groq'")

print(f"Using LLM provider: {LLM_PROVIDER}, model: {ACTIVE_MODEL}")

offices_list = "\n".join([f"- {office}" for office in compensation_offices])

Using LLM provider: groq, model: llama-3.1-8b-instant


In [ ]:
# function to get LLM matches for a batch of query names, with error handling and retries
def get_llm_match_batch(query_names, offices_list, max_retries=3):
    
    queries = "\n".join([f"{i+1}. \"{name}\"" for i, name in enumerate(query_names)])
    
    prompt = f"""You are an expert at matching German compensation office names (Entschädigungsämter).

Given these query names:
{queries}

Find the BEST match for EACH from this list:
{offices_list}

Rules:
1. Return ONLY a numbered list with the exact name from the list
2. Consider spelling variations and OCR errors
3. If no match exists, write \"NO_MATCH\"
4. Format: \"1. Matching Name\" or \"1. NO_MATCH\"

Answers:"""

    messages = [{"role": "user", "content": prompt}]
    
    for attempt in range(max_retries):
        try:
            if LLM_PROVIDER == "huggingface":
                response = client.chat_completion(
                    messages=messages,
                    max_tokens=500,
                    temperature=0.1
                )
                text = response.choices[0].message.content
            elif LLM_PROVIDER == "groq":
                response = client.chat.completions.create(
                    model=ACTIVE_MODEL,
                    messages=messages,
                    max_tokens=500,
                    temperature=0.1
                )
                text = response.choices[0].message.content
            else:
                raise ValueError(f"Unsupported LLM provider: {LLM_PROVIDER}")
            
            matches = []
            lines = text.split('\n')
            for line in lines:
                line = line.strip()
                if line and len(line) > 0 and line[0].isdigit():
                    parts = line.split('.', 1) if '.' in line else line.split(')', 1)
                    if len(parts) > 1:
                        match = parts[1].strip()
                        matches.append(match)
            
            # if the model returned fewer matches than query names, fill the rest with "PARSE_ERROR"
            # TODO: here the mapping might break if one input is skipped by the model in the middle rather than the end
            while len(matches) < len(query_names):
                matches.append("PARSE_ERROR")
            
            return matches[:len(query_names)]
            
        except HfHubHTTPError as e:
            error_str = str(e).lower()
            if "429" in str(e) or "rate" in error_str:
                print(f"Rate limited. Waiting 30s before retry (attempt {attempt+1}/{max_retries})...")
                time.sleep(30)
            elif "402" in str(e) or "quota" in error_str or "exceeded" in error_str:
                print("API quota exceeded! Stopping.")
                return None  # Signal to stop processing
            elif "503" in str(e) or "loading" in error_str:
                print(f"Model loading. Waiting 20s...")
                time.sleep(20)
            else:
                print(f"HTTP Error: {e}")
                if attempt < max_retries - 1:
                    time.sleep(10)
                else:
                    return ["ERROR"] * len(query_names)
        except Exception as e:
            error_str = str(e).lower()
            if "429" in error_str or "rate" in error_str:
                print(f"Rate limited. Waiting 30s before retry (attempt {attempt+1}/{max_retries})...")
                time.sleep(30)
            elif "402" in error_str or "quota" in error_str or "exceeded" in error_str:
                print("API quota exceeded! Stopping.")
                return None  
            elif "503" in error_str or "loading" in error_str:
                print("Model loading. Waiting 20s...")
                time.sleep(20)
            else:
                print(f"Error: {e}")
                if attempt < max_retries - 1:
                    time.sleep(10)
                else:
                    return ["ERROR"] * len(query_names)
    
    return ["ERROR"] * len(query_names)

# test the function with the first 2 unmatched office names
test_names = no_matched_office_df['CompensationOffice1'].iloc[:2].tolist()
print(f"Testing with:")
for i, name in enumerate(test_names):
    print(f"  {i+1}. {name}")
print()
results = get_llm_match_batch(test_names, offices_list)
if results:
    print("LLM suggested matches:")
    for i, result in enumerate(results):
        print(f"  {i+1}. {result}")
else:
    print("API limit reached during test!")

Testing with:
  1. hess. staatsministerium
  2. nan

Rate limited. Waiting 30s before retry (attempt 1/3)...
Rate limited. Waiting 30s before retry (attempt 2/3)...
Rate limited. Waiting 30s before retry (attempt 3/3)...
LLM suggested matches:
  1. ERROR
  2. ERROR


In [ ]:
BATCH_SIZE = 10
DELAY_BETWEEN_BATCHES = 5  

output_file = 'llm_matches_progress.jsonl'

# Load already processed entries to avoid duplicates and allow resuming
processed = set()
llm_matches = []
try:
    with open(output_file, 'r') as f:
        for line in f:
            entry = json.loads(line)
            llm_matches.append(entry)
            processed.add(entry['CompensationOffice1'])
    print(f"Resuming from {len(processed)} already processed entries")
except FileNotFoundError:
    print("Starting fresh")

df_remaining = no_matched_office_df[~no_matched_office_df['CompensationOffice1'].isin(processed)]
print(f"Remaining to process: {len(df_remaining)}")


# Process in batches and save progress after each batch
total_batches = (len(df_remaining) + BATCH_SIZE - 1) // BATCH_SIZE
api_limit_reached = False

for batch_idx in tqdm(range(total_batches), desc="Processing batches"):
    if api_limit_reached:
        break
        
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(df_remaining))
    
    batch_df = df_remaining.iloc[start_idx:end_idx]
    batch_names = batch_df['CompensationOffice1'].tolist()
    
    batch_results = get_llm_match_batch(batch_names, offices_list)
    
    if batch_results is None:
        print("\nAPI limit reached! Saving progress and stopping.")
        api_limit_reached = True
        break
    
    for i, (idx, row) in enumerate(batch_df.iterrows()):
        entry = {
            'CompensationOffice1': row['CompensationOffice1'],
            'MatchedOffice': row['MatchedOffice'],
            'MatchDistance': float(row['MatchDistance']) if pd.notna(row['MatchDistance']) else None,
            'LLMMatch': batch_results[i] if i < len(batch_results) else "ERROR"
        }
        llm_matches.append(entry)
        
        with open(output_file, 'a') as f:
            f.write(json.dumps(entry) + '\n')
    
    if batch_idx < total_batches - 1 and not api_limit_reached:
        time.sleep(DELAY_BETWEEN_BATCHES)

df_llm_matches = pd.DataFrame(llm_matches)
print(f"\nProcessed: {len(df_llm_matches)} entries")
if api_limit_reached:
    print("Note: Processing stopped due to API limit. Run again later to continue.")
df_llm_matches.head(20)

Resuming from 59 already processed entries
Remaining to process: 70457


Processing batches:   0%|          | 0/7046 [00:00<?, ?it/s]

Rate limited. Waiting 30s before retry (attempt 1/3)...
Rate limited. Waiting 30s before retry (attempt 2/3)...
Rate limited. Waiting 30s before retry (attempt 3/3)...


Processing batches:   0%|          | 1/7046 [01:35<186:25:28, 95.26s/it]

Rate limited. Waiting 30s before retry (attempt 1/3)...
Rate limited. Waiting 30s before retry (attempt 2/3)...
Rate limited. Waiting 30s before retry (attempt 3/3)...


...

Rate limited. Waiting 30s before retry (attempt 1/3)...
Rate limited. Waiting 30s before retry (attempt 2/3)...
Rate limited. Waiting 30s before retry (attempt 3/3)...


Processing batches:   2%|▏         | 116/7046 [3:04:18<184:08:59, 95.66s/it]

Rate limited. Waiting 30s before retry (attempt 1/3)...
Rate limited. Waiting 30s before retry (attempt 2/3)...
Rate limited. Waiting 30s before retry (attempt 3/3)...


Processing batches:   2%|▏         | 117/7046 [3:05:54<184:06:30, 95.65s/it]

Rate limited. Waiting 30s before retry (attempt 1/3)...
Rate limited. Waiting 30s before retry (attempt 2/3)...


In [ ]:
valid_matches = df_hf_matches[~df_hf_matches['LLMMatch'].isin(['NO_MATCH', 'ERROR', 'PARSE_ERROR', 'RATE_LIMITED'])]
no_matches = df_hf_matches[df_hf_matches['LLMMatch'] == 'NO_MATCH']
errors = df_hf_matches[df_hf_matches['LLMMatch'].isin(['ERROR', 'PARSE_ERROR', 'RATE_LIMITED'])]

print(f"Total processed: {len(df_hf_matches)}")
print(f"LLM found matches: {len(valid_matches)} ({len(valid_matches)/max(len(df_hf_matches),1)*100:.1f}%)")
print(f"No match found: {len(no_matches)}")
print(f"Errors: {len(errors)}")